# Data Cleaning

In [48]:
import pandas as pd

df = pd.read_csv("./data/spam_ham_dataset.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5171 entries, 0 to 5170
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  5171 non-null   int64
 1   label       5171 non-null   str  
 2   text        5171 non-null   str  
 3   label_num   5171 non-null   int64
dtypes: int64(2), str(2)
memory usage: 161.7 KB


In [49]:
df.head()

,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\r\nthis deal is t...,0


In [50]:
df['label'].value_counts()

label
ham     3672
spam    1499
Name: count, dtype: int64

In [51]:
df['text'].duplicated().sum()

np.int64(178)

In [52]:
pd.crosstab(df["label"], df["label_num"])

label_num,0,1
label,,
ham,3672,0
spam,0,1499


In [53]:
# remove the unnamed col
df = df.drop(columns=["Unnamed: 0"])

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5171 entries, 0 to 5170
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   label      5171 non-null   str  
 1   text       5171 non-null   str  
 2   label_num  5171 non-null   int64
dtypes: int64(1), str(2)
memory usage: 121.3 KB


In [54]:
df.drop_duplicates(subset=["text"], inplace=True)

df.shape

(4993, 3)

In [55]:
df["text"].duplicated().sum()

np.int64(0)

In [56]:
df["text"].iloc[0]

"Subject: enron methanol ; meter # : 988291\r\nthis is a follow up to the note i gave you on monday , 4 / 3 / 00 { preliminary\r\nflow data provided by daren } .\r\nplease override pop ' s daily volume { presently zero } to reflect daily\r\nactivity you can obtain from gas control .\r\nthis change is needed asap for economics purposes ."

# Data Preprocessing

In [57]:
import re

def clean_data(txt: str):
    txt = txt.lower()
    txt = re.sub(r"^subject:\s*", "", txt)
    txt = re.sub(r"[\r\n\t]"," ", txt)
    txt = re.sub(r"[^\w\s]", "", txt)
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt
    
df['clean_text'] = df['text'].apply(clean_data)
df[["text", "clean_text"]].head()

,text,clean_text
0,Subject: enron methanol ; meter # : 988291\r\n...,enron methanol meter 988291 this is a follow u...
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",hpl nom for january 9 2001 see attached file h...
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",neon retreat ho ho ho we re around to that mos...
3,"Subject: photoshop , windows , office . cheap ...",photoshop windows office cheap main trending a...
4,Subject: re : indian springs\r\nthis deal is t...,re indian springs this deal is to book the tec...


In [58]:
print(df["text"].iloc[0])
print()
print(df["clean_text"].iloc[0])

Subject: enron methanol ; meter # : 988291
this is a follow up to the note i gave you on monday , 4 / 3 / 00 { preliminary
flow data provided by daren } .
please override pop ' s daily volume { presently zero } to reflect daily
activity you can obtain from gas control .
this change is needed asap for economics purposes .

enron methanol meter 988291 this is a follow up to the note i gave you on monday 4 3 00 preliminary flow data provided by daren please override pop s daily volume presently zero to reflect daily activity you can obtain from gas control this change is needed asap for economics purposes


# Text to Vector and Model Tranining

In [59]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['label_num']

x_train, x_test, y_train, y_test = train_test_split(X,y,
                                                    stratify=y, 
                                                    test_size=0.2, 
                                                    random_state=42,
                                                   )

In [60]:
y_train.value_counts(normalize=True)

label_num
0    0.707311
1    0.292689
Name: proportion, dtype: float64

In [61]:
y_test.value_counts(normalize=True)

label_num
0    0.706707
1    0.293293
Name: proportion, dtype: float64

In [62]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)

x_train.shape

(3994, 44079)

In [63]:
x_test.shape

(999, 44079)

In [64]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(x_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [65]:
y_pred = model.predict(x_test)
model.score(x_test, y_test)

0.978978978978979

In [66]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99       706
           1       0.96      0.97      0.96       293

    accuracy                           0.98       999
   macro avg       0.97      0.98      0.97       999
weighted avg       0.98      0.98      0.98       999



In [68]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[694  12]
 [  9 284]]


In [45]:
import joblib

joblib.dump(model, "model/spam_model.pkl")
joblib.dump(vectorizer, "model/spam_model_vectorizer.pkl")

['model/spam_model_vectorizer.pkl']

In [69]:
loaded_model = joblib.load("model/spam_model.pkl")
loaded_vectorizer = joblib.load("model/spam_model_vectorizer.pkl")